In [4]:
import nibabel as nib
import numpy as np
import os
from tqdm import tqdm

In [18]:
def load_nifti_file(file_path):
    """Load a NIfTI file and return the image data as a numpy array."""
    img = nib.load(file_path)
    data = img.get_fdata()
    return data



def save_nifti_min_size(np_array, file_path, affine=None, dtype=np.float32):
    """
    Save a NumPy array as a compressed NIfTI file with minimum size.

    Parameters
    ----------
    np_array : np.ndarray
        The image data to save.
    file_path : str
        Output path for .nii.gz file (recommended).
    affine : np.ndarray, optional
        4×4 affine matrix. If None, identity is used.
    dtype : np.dtype
        Data type to save as (e.g., np.float32, np.uint8).
    """

    # Use identity affine if none provided
    if affine is None:
        affine = np.eye(4)

    # Convert array dtype to reduce size
    np_array = np.asarray(np_array).astype(dtype)

    # Create and save compressed NIfTI
    nifti_img = nib.Nifti1Image(np_array, affine)

    if not file_path.endswith(".gz"):
        file_path += ".gz"  # enforce gzip compression

    nib.save(nifti_img, file_path)
    print(f"Saved compressed NIfTI file (dtype={dtype}) to: {file_path}")



def read_json(file_path):
    """Read a JSON file and return its content as a dictionary."""
    import json
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

def iterate_over_all_files(root_dir):
    data_directories_dir = os.path.join(root_dir, "dataset_directories.json")
    dict_dataset = read_json(data_directories_dir)
    
    center_1_root = os.path.join(root_dir, "Masih-SUV")
    center_1_dicts = dict_dataset.get("Masih-SUV", [])
    for item in tqdm(center_1_dicts, desc="Processing Center 1 Files"):
        sample_dir_name, sample_info = list(item.items())[0]
        img_name = sample_info['image']
        img_dir = os.path.join(center_1_root, sample_dir_name, img_name.replace('.nii', '_prep_img.nii.gz'))
        seg_name = sample_info['segmentation']
        seg_dir = os.path.join(center_1_root, sample_dir_name, seg_name.replace('.nrrd', '_prep_seg.nii.gz'))
        img_data = load_nifti_file(img_dir)
        seg_data = load_nifti_file(seg_dir)
        hl_label = 0 if sample_info['Binary Label'] == 'NHL' else 1
        yield seg_data, hl_label, img_data
    
def find_max_dimensions(root_dir):
    max_dims = [0, 0, 0]
    for seg_data, _, _ in iterate_over_all_files(root_dir):
        current_dims = seg_data.shape
        for i in range(3):
            if current_dims[i] > max_dims[i]:
                max_dims[i] = current_dims[i]
    return max_dims



In [15]:
root_dir = "/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/new_dataset"

In [7]:
max_size = find_max_dimensions(root_dir)
print("Maximum dimensions across all segmentation files:", max_size)

Processing Center 1 Files: 100%|██████████| 151/151 [00:20<00:00,  7.52it/s]

Maximum dimensions across all segmentation files: [224, 224, 371]


In [16]:
def nodes_heatmap(root_dir, max_size=[224,224,371]):
    hodgkin_heatmap = np.zeros(max_size)
    non_hodgkin_heatmap = np.zeros(max_size)
    image_heatmap = np.zeros(max_size)
    hodgkin_num, non_hodgkin_num = 0, 0
    for seg_data, hl_label, img_data in iterate_over_all_files(root_dir):
        if hl_label == 1:
            hodgkin_num += 1
            hodgkin_heatmap[:seg_data.shape[0], :seg_data.shape[1], :seg_data.shape[2]] += (seg_data > 0).astype(np.uint)
        else:
            non_hodgkin_num += 1
            non_hodgkin_heatmap[:seg_data.shape[0], :seg_data.shape[1], :seg_data.shape[2]] += (seg_data > 0).astype(np.uint8)
        image_heatmap[:img_data.shape[0], :img_data.shape[1], :img_data.shape[2]] += img_data
    hodgkin_heatmap /= hodgkin_num
    non_hodgkin_heatmap /= non_hodgkin_num
    image_heatmap /= (hodgkin_num + non_hodgkin_num)
    return hodgkin_heatmap, non_hodgkin_heatmap, image_heatmap

hodgkin_heatmap, non_hodgkin_heatmap, image_heatmap = nodes_heatmap(root_dir, max_size)

Processing Center 1 Files: 100%|██████████| 151/151 [01:43<00:00,  1.45it/s]


In [19]:
os.makedirs("location_heatmaps", exist_ok=True)
save_nifti_min_size(hodgkin_heatmap, "location_heatmaps/hodgkin_heatmap.nii.gz")
save_nifti_min_size(non_hodgkin_heatmap, "location_heatmaps/non_hodgkin_heatmap.nii.gz")
save_nifti_min_size(image_heatmap, "location_heatmaps/image_heatmap.nii.gz")

Saved compressed NIfTI file (dtype=<class 'numpy.float32'>) to: location_heatmaps/hodgkin_heatmap.nii.gz
Saved compressed NIfTI file (dtype=<class 'numpy.float32'>) to: location_heatmaps/non_hodgkin_heatmap.nii.gz
Saved compressed NIfTI file (dtype=<class 'numpy.float32'>) to: location_heatmaps/image_heatmap.nii.gz
